# Notebook 0b: three ways to bring data in, and feed it to a model

Notebook 0 made its data inside the notebook. Real data comes from somewhere else. This notebook shows the three ways you will meet most often, then feeds each one into the same model:

| Part | Where the data comes from | Example |
|---|---|---|
| **A** | **Excel files from a GitHub repository** | the plant's influent files, as our project gets them |
| **B** | **A file you make yourself (synthetic data)** | a made-up sensor, saved as CSV and read back |
| **C** | **A link on the internet** | 10 years of daily temperatures, fetched from a web address |
| **D** | **Feed any of them to a model** | the same function for all three |

Whatever the source, the goal is the same: **one pandas table (DataFrame) with the right types**. After that, every project is the same.

Run the cells from top to bottom with **Shift + Enter**.

In [ ]:
from pathlib import Path
from urllib.request import urlretrieve
import numpy as np
import pandas as pd
import torch
from torch import nn

DATA = Path("../data")          # the data folder next to this notebook
DATA.mkdir(exist_ok=True)

## A. Excel files from a GitHub repository

Our project's data, the official BSM1 influent files, lives in someone else's GitHub repository (`davidfernand/BSM1_in_Python`). There are two ways to get files from GitHub:

1. **Clone the whole repository** in a terminal: `git clone https://github.com/davidfernand/BSM1_in_Python.git`. Then the files are on your disk and you read them by their path.
2. **Download just the file you need** from its "raw" address. This is what the project does.

The address contains a **commit hash** (`5dadb20…`): it points to one fixed version of the file, so anyone can get exactly the same data years later.

In [ ]:
REV = "5dadb209a5a34ecb0e2f83a2e79c060bf3470582"         # one fixed version of the repository
GITHUB = f"https://raw.githubusercontent.com/davidfernand/BSM1_in_Python/{REV}/code/Influent%20files/"

xlsx = DATA / "Inf_dry_2006.xlsx"
if not xlsx.exists():                                    # download only the first time
    urlretrieve(GITHUB + xlsx.name, xlsx)
print(xlsx, xlsx.stat().st_size // 1000, "kB")

Now read it. **Look first**: read it raw, without telling pandas anything.

In [ ]:
raw = pd.read_excel(xlsx, header=None)
print(raw.shape)
print(raw.dtypes.head(4))          # object = text: something in each column is not a number
raw.head(3)

Row 0 holds the column names, so pandas read every column as text. Tell it row 0 is the header, give the columns names we understand, and make them numbers.

In [ ]:
COLS = ["t", "S_I", "S_S", "X_I", "X_S", "X_BH", "X_BA", "X_P", "S_O", "S_NO", "S_NH", "S_ND", "X_ND", "S_ALK", "Q"]
plant = pd.read_excel(xlsx, header=0)
plant.columns = COLS
plant = plant.astype(float)                 # fails loudly if anything is not a number: good
print(plant.shape, "  time step:", np.unique(np.round(np.diff(plant.t) * 1440, 3)), "minutes")
plant[["t", "Q", "S_NH", "S_S"]].head()

Excel options you will need: `sheet_name=` (which sheet, by name or number), `skiprows=` (title lines at the top), `usecols="A:C"` (some columns only). `pd.ExcelFile(path).sheet_names` lists the sheets.

## B. A file you make yourself (synthetic data)

Synthetic data is data you generate with a known rule. It is useful to test your code before the real data arrives, because **you know the right answer**. Here: a sensor reading every hour for 60 days, with a daily cycle, a slow trend and noise.

We save it to a CSV file, the way an instrument or a colleague would give it to you, and then read it back as if it came from outside.

In [ ]:
rng = np.random.default_rng(0)
hours = pd.date_range("2025-01-01", periods=60 * 24, freq="h")
h = np.arange(len(hours))
reading = 50 + 10 * np.sin(2 * np.pi * h / 24) + 0.01 * h + rng.normal(0, 1.5, len(h))

pd.DataFrame({"time": hours, "reading": reading.round(2)}).to_csv(DATA / "synthetic_sensor.csv", index=False)

# ...later, or on another computer: read the file
sensor = pd.read_csv(DATA / "synthetic_sensor.csv", parse_dates=["time"], index_col="time")
print(sensor.shape, sensor.dtypes.to_dict())
sensor.head()

CSV options for untidy files: `sep=";"` (another separator), `decimal=","` (European numbers), `skiprows=2` (title lines), `na_values=["-", "NA"]` (how missing values are written), `encoding="latin-1"` (strange characters).

## C. A link on the internet

`read_csv` also takes a web address. Here: daily minimum temperatures in Melbourne, 1981 to 1990, a small public dataset. Two habits: **save a local copy** (the website may change or disappear), and use the copy when there is no internet.

In [ ]:
LINK = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/daily-min-temperatures.csv"
local = DATA / "daily-min-temperatures.csv"
try:
    temps = pd.read_csv(LINK, parse_dates=["Date"], index_col="Date")
    temps.to_csv(local)                                   # keep a copy
    print("fetched from the link:", temps.shape)
except Exception as e:
    temps = pd.read_csv(local, parse_dates=["Date"], index_col="Date")
    print("no internet, read the saved copy:", temps.shape, type(e).__name__)
temps.describe().round(1)

Other places data comes from, all ending in a DataFrame the same way:
- **Google Sheet shared "anyone with the link":** `pd.read_csv("https://docs.google.com/spreadsheets/d/<SHEET_ID>/export?format=csv")`
- **Google Drive in Colab:** `from google.colab import drive; drive.mount("/content/drive")`, then a normal path.
- **Kaggle:** download and unzip the dataset, then part B's `read_csv`.
- **A database:** `pd.read_sql("SELECT ... FROM ...", connection)`.
- **A web API:** `json.load(urllib.request.urlopen(url))`, then `pd.DataFrame(...)` of the part you need.

## D. Feed it to a model: one function for all three

From here the source no longer matters. The function below takes **any series of numbers in time order** and does the steps from notebook 0:

1. **Inputs and target:** X = the last 8 values, y = the next value.
2. **Split by time:** 70 % train, 15 % validation, 15 % test.
3. **Scale** with the training numbers only.
4. **Feed:** tensors and batches (DataLoader).
5. **Model and training loop,** keeping the best validation epoch.
6. **Test once** against the simplest guess: "next = now".

In [ ]:
import copy
from torch.utils.data import TensorDataset, DataLoader

def feed_to_model(values, name, lookback=8, epochs=40):
    v = np.asarray(values, dtype=np.float32)
    # 1. inputs and target: sliding windows
    X = np.stack([v[i:i + lookback] for i in range(len(v) - lookback)])     # (N, lookback)
    y = v[lookback:].reshape(-1, 1)                                         # (N, 1)
    # 2. split by time
    n = len(X); a, b = int(0.7 * n), int(0.85 * n)
    # 3. scale with training numbers only
    mean, std = X[:a].mean(), X[:a].std()
    s = lambda z: torch.tensor((z - mean) / std, dtype=torch.float32)
    # 4. feed: tensors and batches
    train_dl = DataLoader(TensorDataset(s(X[:a]), s(y[:a])), batch_size=64, shuffle=True)
    Xv, yv = s(X[a:b]), s(y[a:b])
    # 5. model and training loop
    torch.manual_seed(0)
    model = nn.Sequential(nn.Linear(lookback, 16), nn.ReLU(), nn.Linear(16, 1))
    opt, loss_fn = torch.optim.Adam(model.parameters(), lr=1e-2), nn.MSELoss()
    best, best_state = float("inf"), None
    for epoch in range(epochs):
        model.train()
        for xb, yb in train_dl:
            loss = loss_fn(model(xb), yb)
            opt.zero_grad(); loss.backward(); opt.step()
        model.eval()
        with torch.no_grad():
            val = loss_fn(model(Xv), yv).item()
        if val < best:
            best, best_state = val, copy.deepcopy(model.state_dict())
    model.load_state_dict(best_state)
    # 6. test once, in real units, against "next = now"
    with torch.no_grad():
        pred = model(s(X[b:])).numpy() * std + mean
    mae_model = np.abs(pred - y[b:]).mean()
    mae_base = np.abs(X[b:, -1:] - y[b:]).mean()
    print(f"{name:32s} X {X.shape}  test error: model {mae_model:8.2f}   baseline {mae_base:8.2f}")

In [ ]:
feed_to_model(plant["Q"], "A · plant inflow Q (Excel, GitHub)")
feed_to_model(sensor["reading"], "B · synthetic sensor (CSV file)")
feed_to_model(temps["Temp"], "C · Melbourne temperature (link)")

Three sources, one pipeline. In all three the model beats "next = now", but only by a little (plant flow about 824 against 843 m³/d, synthetic 2.2 against 2.4, temperature 1.7 against 2.0 °C). Without the baseline you could not tell whether the model is any good: that is why every test includes one.

**To use your own data:** load it with part A, B or C, pick the column, and call `feed_to_model(your_df["your_column"], "my data")`.